# Option 1 experiment: continue LoRA training from epoch3 for 2 more epochs

Starts from the S5 `epoch3` adapter (frozen as the main result in results/threshold.md; this does NOT replace it),
fresh AdamW + cosine schedule at lr 5e-5, a different shuffle seed, same data (train_v2). Outputs
`/kaggle/working/out/lora_ext/epoch{1,2}` (= total epochs 4 and 5). Dev evaluation is a separate step.

Needs: GPU, Internet, secret `HF_TOKEN`; inputs `agentic-distill-src`, `agentic-distill-sft`, output of `agentic-distill-s5-train`.


In [ ]:
import sys, subprocess, os, glob, re, hashlib
print(sys.version)
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)
SFT = glob.glob("/kaggle/input/**/train_v2.jsonl", recursive=True); assert SFT, "attach agentic-distill-sft"
hits = glob.glob("/kaggle/input/**/lora/epoch3/adapter_config.json", recursive=True); assert hits, "attach s5-train output"
INIT = os.path.dirname(hits[0]); print("init adapter:", INIT)
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")   # value never printed
print("HF_TOKEN loaded:", bool(os.environ["HF_TOKEN"]))


In [ ]:
%%bash
set -e
rm -rf /kaggle/working/repo && mkdir -p /kaggle/working/repo
D=$(find /kaggle/input -maxdepth 3 -type d -name agentic-distill-src | head -1)
echo "code dataset dir: $D"
Z=$(find "$D" -name agentic_distill_src.zip | head -1)
if [ -n "$Z" ]; then (cd /kaggle/working/repo && unzip -q "$Z"); else cp -r "$D"/. /kaggle/working/repo/; fi
cd /kaggle/working/repo
grep -q -- "--init-adapter" src/agentic_distill/train_lora.py
pip uninstall -y -q torchao
pip install -q -U transformers peft accelerate safetensors
python -c "import torch, transformers, peft; print('torch', torch.__version__, 'transformers', transformers.__version__, 'peft', peft.__version__, 'gpus', torch.cuda.device_count())"


In [ ]:
# P1 guard: the starting adapter must be the one frozen in results/threshold.md
frozen = re.search(r"sha256 `([0-9a-f]{64})`", open("/kaggle/working/repo/results/threshold.md").read()).group(1)
h = hashlib.sha256(open(os.path.join(INIT, "adapter_model.safetensors"), "rb").read()).hexdigest()
print("epoch3 sha256:", h); assert h == frozen


In [ ]:
env = {**os.environ, "PYTHONPATH": "src", "CUDA_VISIBLE_DEVICES": "0", "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True"}
cmd = ["python", "-m", "agentic_distill.train_lora", "--data", SFT[0], "--out", "/kaggle/working/out/lora_ext",
       "--init-adapter", INIT, "--epochs", "2", "--lr", "5e-5", "--seed", "20261116", "--resume"]
print(" ".join(cmd), flush=True)
p = subprocess.Popen(cmd, cwd="/kaggle/working/repo", env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout: print(line, end="", flush=True)
assert p.wait() == 0, f"train_lora exited {p.returncode}"
